# Document Loader

将不同来源的数据加载为统一的 `Document`：`page_content` 保存正文，`metadata` 保存来源等信息。

## 准备

先选择项目的 Python 内核，运行下面的单元格安装依赖。安装后如遇到导入问题，重启内核再继续。

PDF 使用项目已有的 `pypdf`。示例文件位于 `resource/document_loader/`，PDF 复用 `resource/ECMA.pdf`。

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"`langchain-community` is being sunset.*",
    category=DeprecationWarning,
)

In [ ]:
import sys

sys.executable

In [ ]:
import sys

# 安装到当前 Notebook 内核使用的 Python 环境
!uv pip install --python "{sys.executable}" langchain-community "unstructured[md]" jq

In [ ]:
from pathlib import Path

# 兼容从项目根目录或课件目录启动 Notebook
project_dir = Path.cwd()
if not (project_dir / "pyproject.toml").exists():
    project_dir = project_dir.parent

resource_dir = project_dir / "resource"
sample_dir = resource_dir / "document_loader"

## 纯文本：TextLoader

整个文本文件加载为一个 `Document`，指定 UTF-8 编码。

In [ ]:
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader(sample_dir / "sales_policy.txt", encoding="utf-8")
text_docs = text_loader.load()

len(text_docs)

In [ ]:
print(text_docs[0].page_content)
text_docs[0].metadata

## Markdown：UnstructuredMarkdownLoader

`mode="elements"` 按标题、段落、列表项等元素输出多个 `Document`。正文是解析后的文本，元素类别记录在 `metadata["category"]` 中，不保留原始 Markdown 标记。

首次运行时，Unstructured 会自动下载其 spaCy 文本分析模型，需要联网。

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

markdown_loader = UnstructuredMarkdownLoader(
    str(sample_dir / "sales_faq.md"),
    mode="elements",
)
markdown_docs = markdown_loader.load()

len(markdown_docs)

In [ ]:
for doc in markdown_docs:
    print(doc.metadata.get("category"), "|", doc.page_content)

markdown_docs[0].metadata

## CSV：CSVLoader

每条数据行生成一个 `Document`，列名和值组成正文；`metadata["row"]` 从 0 开始。

In [11]:
from langchain_community.document_loaders import CSVLoader

csv_loader = CSVLoader(
    file_path=str(sample_dir / "products.csv"),
    encoding="utf-8",
)
csv_docs = csv_loader.load()

len(csv_docs)

12

In [12]:
print(csv_docs[0].page_content)
csv_docs[0].metadata

产品名称: 标准版
部署方式: 云端部署
适用客户: 需要快速搭建内部知识库的小型团队，以产品资料查询和常见问题问答为主要场景。


{'source': '/Volumes/数据盘/课程资料/AI/deal-pilot-rag/resource/document_loader/products.csv',
 'row': 0}

## JSON：JSONLoader

`jq_schema=".faqs[].answer"` 提取每条问答的 `answer`，每个答案生成一个 `Document`；默认元数据包含文件来源和从 1 开始的 `seq_num`。

In [13]:
from langchain_community.document_loaders import JSONLoader

json_loader = JSONLoader(
    file_path=sample_dir / "faq.json",
    jq_schema=".faqs[].answer",
    text_content=True,
)
json_docs = json_loader.load()

len(json_docs)

12

In [14]:
print(json_docs[0].page_content)
json_docs[0].metadata

标准版适合小型团队，提供知识库问答和基础数据分析。客户可以先选择产品资料查询或常见问题问答等单一场景开展试点，确认使用效果后再逐步扩大范围。本文件中的产品权益和服务约定均为课程演示内容。


{'source': '/Volumes/数据盘/课程资料/AI/deal-pilot-rag/resource/document_loader/faq.json',
 'seq_num': 1}

## PDF：PyPDFLoader

`mode="page"` 每页生成一个 `Document`，`metadata["page"]` 从 0 开始。

示例通过 `load()` 加载全部页面。这里提取 PDF 文本层；扫描件的识字需要另行配置 OCR。

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader(str(resource_dir / "ECMA.pdf"), mode="page")
pdf_docs = pdf_loader.load()

len(pdf_docs)

328

In [17]:
print(pdf_docs[100].page_content[:800])
pdf_docs[100].metadata

解构的参数是必需的
参数解构有一个怪异点：默认情况下调用函数时未给参数解构传值会抛出错误。例如，用以
下方式调用上例中的
	
	
setCookie()
	
	
函数就会出错：
One	quirk	of	using	destructured	parameters	is	that,	by	default,	an	error	is	thrown	when	they
are	not	provided	in	a	function	call.	For	instance,	this	call	to	the	
	
setCookie()
	
	function	in	the
last	example	throws	an	error:
//	
出错！
setCookie(
"type"
,	
"js"
);
调用时第三个参数缺失了，因此它不出预料地等于
	
	
undefined
	
	
。这导致了一个错误，因为参
数解构实际上只是解构声明的简写。当
	
	
setCookie()
	
	
函数被调用时，
	JS	
引擎实际上是这么
做的：
function
	
setCookie
(
name,	value,	options
)	
{
				
let
	{	secure,	path,	domain,	expires	}	= 	options;
				
//	
设置
	cookie	
的代码
}
既然在赋值右侧的值为
	
	
null
	
	
或
	
	
undefined
	
	
时，解构会抛出错误，那么未向
	
	
setCookie()
	
函数传递第三个参数就同样会出错。
若你让解构的参数作为必选参数，那么上述行为并不会令人困扰。但若你要求它是可选的，
可以给解构的参数提供默认值来处理这种行为，就像这样：
function
	
setCookie
(
name,	value,	


{'producer': 'calibre 2.57.1 [http://calibre-ebook.com]',
 'creator': 'calibre 2.57.1 [http://calibre-ebook.com]',
 'creationdate': '2017-03-26T17:39:43+00:00',
 'author': 'sagittarius-rev',
 'title': '《Understanding ECMAScript 6》中文翻译',
 'source': '/Volumes/数据盘/课程资料/AI/deal-pilot-rag/resource/ECMA.pdf',
 'total_pages': 328,
 'page': 100,
 'page_label': '101'}

## 参考

- [LangChain Document Loaders](https://docs.langchain.com/oss/python/integrations/document_loaders/)
- [Community Loader API](https://reference.langchain.com/python/langchain-community/document_loaders)